In [ ]:
import pandas as pd
import numpy as np
import csv
%load_ext rpy2.ipython


In [ ]:
pip install rpy2


In [ ]:


%%R
install.packages("VIM")


In [ ]:
%%R

required_packages <- c("tidyverse", "randomForest", "ggplot2", "corrplot", "VIM", "mice", "car", "effsize", 
                       "knitr", "kableExtra", "plotly", "DT", "gridExtra", "RColorBrewer", "viridis", 
                       "broom", "lme4", "sjPlot", "factoextra", "cluster")

to_install <- setdiff(required_packages, rownames(installed.packages()))
if(length(to_install)) install.packages(to_install)


# stats in R

In [ ]:
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import pandas2ri
import rpy2.robjects as ro


with localconverter(ro.default_converter + pandas2ri.converter):
    r_df = ro.conversion.py2rpy(combined)


In [ ]:

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

# Example: Convert a pandas DataFrame to R dataframe
with localconverter(ro.default_converter + pandas2ri.converter):
    r_df = ro.conversion.py2rpy(combined)

# Assign it to the R global environment if needed
ro.globalenv['combined'] = r_df


# MCR 

In [ ]:
%R -i df

In [ ]:
%%R

library(dplyr)
library(ggplot2)
library(plotly)
library(FactoMineR)
library(factoextra)
library(cluster)


lesion_counts <- df %>%
  group_by(record_id) %>%
  summarise(
    n_lesions = n(),
    n_bilateral = sum(`bilateral?` == 1, na.rm = TRUE),
    .groups = "drop"
  )


In [ ]:

patient_data <- df %>%
  group_by(record_id) %>%
  slice(1) %>%
  ungroup() %>%
  left_join(lesion_counts, by = "record_id")

patient_features <- patient_data

cat(sprintf("Starting with %d patients\n", nrow(patient_features)))

# Prepare variables for MCA

# Separate metadata from analysis variables
metadata <- patient_features %>%
  select(record_id, site)

analysis_vars <- patient_features %>%
  select(-record_id, -site, 
         -starts_with("biopsy_site"), 
         -starts_with("biopsy_date"),
         -starts_with("onset_of_symptoms"),
         -starts_with("diagnosis_date"),
         -starts_with("mri_date"),
         -starts_with("dob"),
         -contains("other"),
         -contains("complete"),
         -contains("b_values"))

# Process variables for MCA

mca_matrix <- data.frame(record_id = patient_features$record_id)

# A) Binary variables - convert to factors with meaningful labels
binary_cols <- analysis_vars %>%
  select(where(~is.numeric(.) && all(. %in% c(0, 1, NA)))) %>%
  names()

if(length(binary_cols) > 0) {
  cat(sprintf("Processing %d binary variables\n", length(binary_cols)))
  for(col in binary_cols) {
    mca_matrix[[col]] <- factor(analysis_vars[[col]], 
                                levels = c(0, 1), 
                                labels = c("No", "Yes"))
  }
}

# B) Categorical variables - convert to factors
categorical_cols <- c("gender", "vendor", "anesthesia", "coverage", 
                     "biopsy_number", "blood_culture_result", "biopsy_type")

for(col in categorical_cols) {
  if(col %in% names(analysis_vars)) {
    cat(sprintf("Converting to factor: %s\n", col))
    mca_matrix[[col]] <- as.factor(analysis_vars[[col]])
  }
}

# C) Continuous variables - discretize into categories for MCA
continuous_cols <- c("n_lesions", "n_bilateral", "crp_result", 
                    "sed_rate_result", "wbc_count_result",
                    "total_time_of_acquisition", "months_between_diagnosis_and_mri")

for(col in continuous_cols) {
  if(col %in% names(analysis_vars)) {
    cat(sprintf("Discretizing: %s\n", col))
    
    # Create categories based on distribution
    if(col %in% c("n_lesions", "n_bilateral")) {
      vals <- as.numeric(analysis_vars[[col]])
      mca_matrix[[col]] <- cut(vals, 
                              breaks = c(-Inf, 2, 5, Inf),
                              labels = c("Low", "Medium", "High"),
                              include.lowest = TRUE)
    } else if(col == "total_time_of_acquisition") {
      # For acquisition time: quartiles
      vals <- as.numeric(analysis_vars[[col]])
      mca_matrix[[col]] <- cut(vals,
                              breaks = quantile(vals, probs = c(0, 0.33, 0.67, 1), na.rm = TRUE),
                              labels = c("Short", "Medium", "Long"),
                              include.lowest = TRUE)
    } else {
      vals <- as.numeric(analysis_vars[[col]])
      if(sum(!is.na(vals)) > 10) {  
        mca_matrix[[col]] <- cut(vals,
                                breaks = quantile(vals, probs = c(0, 0.33, 0.67, 1), na.rm = TRUE),
                                labels = c("Low", "Medium", "High"),
                                include.lowest = TRUE)
      }
    }
  }
}

# Remove record_id for analysis
mca_ready <- mca_matrix %>% select(-record_id)

cat(sprintf("\nTotal variables before cleaning: %d\n", ncol(mca_ready)))

# Clean data for MCA

# Remove columns with >50% missing data
na_proportion <- colMeans(is.na(mca_ready))
keep_cols <- na_proportion < 0.5
mca_clean <- mca_ready[, keep_cols]

cat(sprintf("After removing high-missing (>50%%): %d variables\n", ncol(mca_clean)))


for(col in names(mca_clean)) {
  if(any(is.na(mca_clean[[col]]))) {
    
    mca_clean[[col]] <- addNA(mca_clean[[col]])
    levels(mca_clean[[col]])[is.na(levels(mca_clean[[col]]))] <- "Missing"
  }
}

# Remove variables with near-zero variance (all same category)
# Calculate category frequencies
remove_cols <- c()
for(col in names(mca_clean)) {
  freq_table <- table(mca_clean[[col]])
  max_freq <- max(freq_table) / sum(freq_table)
  if(max_freq > 0.95) {  
    remove_cols <- c(remove_cols, col)
  }
}

if(length(remove_cols) > 0) {
  cat(sprintf("Removing %d low-variance variables: %s\n", 
              length(remove_cols), paste(remove_cols, collapse = ", ")))
  mca_clean <- mca_clean[, !(names(mca_clean) %in% remove_cols)]
}

cat(sprintf("After removing low-variance: %d variables\n", ncol(mca_clean)))

# Ensure all columns are factors
mca_clean <- mca_clean %>% mutate(across(everything(), as.factor))

cat(sprintf("Final variables for MCA: %d\n", ncol(mca_clean)))

# Run MCA

cat("\n========================================\n")
cat("RUNNING MCA\n")
cat("========================================\n")

# Run MCA
mca_result <- MCA(mca_clean, 
                  ncp = 10,  
                  graph = FALSE)  

var_explained <- mca_result$eig

cat("\n========================================\n")
cat("VARIANCE (INERTIA) EXPLAINED\n")
cat("========================================\n")
cat("Dimension | Eigenvalue | % Variance | Cumulative %\n")
cat("----------|------------|------------|-------------\n")
for(i in 1:min(10, nrow(var_explained))) {
  cat(sprintf("    %2d    |   %6.3f   |   %6.2f   |    %6.2f\n",
              i, var_explained[i,1], var_explained[i,2], var_explained[i,3]))
}

cumvar_3 <- var_explained[3, 3]
cat(sprintf("\nFirst 3 dimensions capture %.1f%% of inertia\n", cumvar_3))

# Scree plot
fviz_screeplot(mca_result, addlabels = TRUE, ylim = c(0, 15), ncp = 15,
               main = "Scree Plot: MCA Variance Explained")


# Extract dimension scores

dim_scores <- data.frame(
  record_id = patient_features$record_id,
  site = patient_features$site,
  Dim1 = mca_result$ind$coord[, 1],
  Dim2 = mca_result$ind$coord[, 2],
  Dim3 = mca_result$ind$coord[, 3]
)

# Examine variable contributions

cat("\n========================================\n")
cat("TOP CONTRIBUTING VARIABLES\n")
cat("========================================\n")

# Get variable contributions
var_contrib <- mca_result$var$contrib

for(dim in 1:4) {
  cat(sprintf("\n--- Dimension %d (%.1f%% variance) ---\n", 
              dim, var_explained[dim, 3]))
  
  # Sort by contribution
  sorted_idx <- order(var_contrib[, dim], decreasing = TRUE)[1:20]
  
  for(idx in sorted_idx) {
    var_name <- rownames(var_contrib)[idx]
    contrib_val <- var_contrib[idx, dim]
    cat(sprintf("  %-50s: %7.3f%%\n", var_name, contrib_val))
  }
}

# Contribution plots
fviz_contrib(mca_result, choice = "var", axes = 1, top = 25,
             title = "Top 25 Variables Contributing to Dim1")

fviz_contrib(mca_result, choice = "var", axes = 2, top = 25,
             title = "Top 25 Variables Contributing to Dim2")

fviz_contrib(mca_result, choice = "var", axes = 3, top = 25,
             title = "Top 25 Variables Contributing to Dim3")

# K-Means clustering on MCA dimensions

# Silhouette analysis
cat("\n========================================\n")
cat("SILHOUETTE ANALYSIS\n")
cat("========================================\n")

sil_scores <- c()
for(k in 2:8) {
  set.seed(123)
  km <- kmeans(dim_scores[, c("Dim1", "Dim2", "Dim3")], centers = k, nstart = 25)
  sil <- silhouette(km$cluster, dist(dim_scores[, c("Dim1", "Dim2", "Dim3")]))
  sil_scores[k-1] <- mean(sil[, 3])
}

names(sil_scores) <- 2:8
print(round(sil_scores, 3))

# Optimal K
optimal_k <- which.max(sil_scores) + 1
cat(sprintf("\nOptimal K = %d (silhouette = %.3f)\n", optimal_k, max(sil_scores)))

# Plot silhouette
plot(2:8, sil_scores, type = "b", pch = 19, col = "steelblue", lwd = 2,
     xlab = "Number of Clusters", ylab = "Avg Silhouette Width",
     main = "Silhouette Analysis", cex = 1.5)
grid()
abline(v = 3, col = "red", lty = 2)

# Run K-means with K=3
set.seed(123)
kmeans_result <- kmeans(dim_scores[, c("Dim1", "Dim2", "Dim3")], 
                       centers = 3, nstart = 25)

dim_scores$cluster <- factor(kmeans_result$cluster)

cat("\n--- K=3 Clustering Results ---\n")
cat("Cluster sizes:\n")
print(table(dim_scores$cluster))
cat(sprintf("Between/Total SS: %.1f%%\n", 
            100 * kmeans_result$betweenss / kmeans_result$totss))

# Visualize results

site_colors <- c("CMH" = "#E41A1C", "NCH" = "#377EB8", "SK" = "#4DAF4A")
cluster_colors <- c("1" = "#984EA3", "2" = "#FF7F00", "3" = "#FFFF33","4" = "#FF2F33")

# MCA variable plot
fviz_mca_var(mca_result, 
             axes = c(1, 2),
             repel = TRUE,
             title = "MCA Variable Categories Map")

# Interactive 3D plot by site
fig_site_interactive <- plot_ly(dim_scores,
  x = ~Dim1, y = ~Dim2, z = ~Dim3,
  color = ~site,
  colors = site_colors,
  type = "scatter3d",
  mode = "markers",
  marker = list(size = 5, opacity = 0.7),
  text = ~paste("ID:", record_id, "<br>Site:", site, "<br>Cluster:", cluster),
  hoverinfo = "text"
) %>%
  layout(
    title = list(text = "MCA by Hospital Site", font = list(size = 16)),
    scene = list(
      xaxis = list(title = sprintf("Dim1 (%.1f%%)", var_explained[1,2])),
      yaxis = list(title = sprintf("Dim2 (%.1f%%)", var_explained[2,2])),
      zaxis = list(title = sprintf("Dim3 (%.1f%%)", var_explained[3,2]))
    )
  )

print(fig_site_interactive)

# Interactive 3D plot by cluster
fig_cluster_interactive <- plot_ly(dim_scores,
  x = ~Dim1, y = ~Dim2, z = ~Dim3,
  color = ~cluster,
  colors = cluster_colors,
  type = "scatter3d",
  mode = "markers",
  marker = list(size = 5, opacity = 0.7),
  text = ~paste("ID:", record_id, "<br>Site:", site, "<br>Cluster:", cluster),
  hoverinfo = "text"
) %>%
  layout(
    title = list(text = "MCA by K-Means Cluster (K=3)", font = list(size = 16)),
    scene = list(
      xaxis = list(title = sprintf("Dim1 (%.1f%%)", var_explained[1,2])),
      yaxis = list(title = sprintf("Dim2 (%.1f%%)", var_explained[2,2])),
      zaxis = list(title = sprintf("Dim3 (%.1f%%)", var_explained[3,2]))
    )
  )

print(fig_cluster_interactive)

# 2D ggplot versions
p1 <- ggplot(dim_scores, aes(x = Dim1, y = Dim2, color = cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  stat_ellipse(level = 0.95, linewidth = 1) +
  scale_color_manual(values = cluster_colors) +
  labs(title = "Dim1 vs Dim2: Clusters",
       x = sprintf("Dim1 (%.1f%%)", var_explained[1,2]),
       y = sprintf("Dim2 (%.1f%%)", var_explained[2,2]),
       color = "Cluster") +
  theme_minimal(base_size = 14) +
  theme(legend.position = "right")

print(p1)

p2 <- ggplot(dim_scores, aes(x = Dim1, y = Dim2, color = site)) +
  geom_point(size = 3, alpha = 0.7) +
  stat_ellipse(level = 0.95, linewidth = 1) +
  scale_color_manual(values = site_colors) +
  labs(title = "Dim1 vs Dim2: Hospital Sites",
       x = sprintf("Dim1 (%.1f%%)", var_explained[1,2]),
       y = sprintf("Dim2 (%.1f%%)", var_explained[2,2]),
       color = "Site") +
  theme_minimal(base_size = 14) +
  theme(legend.position = "right")

print(p2)

#Cluster characterization

cat("\n========================================\n")
cat("CLUSTER vs SITE\n")
cat("========================================\n")

cluster_site <- table(dim_scores$cluster, dim_scores$site)
print(cluster_site)
print(round(prop.table(cluster_site, margin = 1) * 100, 1))

# Chi-square test
chi_test <- chisq.test(cluster_site)
cat(sprintf("\nχ² = %.2f, p = %.4f\n", chi_test$statistic, chi_test$p.value))

# =========================================
# Summary
# =========================================

cat("\n========================================\n")
cat("SUMMARY\n")
cat("========================================\n")
cat(sprintf("✓ %d patients analyzed\n", nrow(dim_scores)))
cat(sprintf("✓ %d variables in MCA\n", ncol(mca_clean)))
cat(sprintf("✓ First 3 dimensions: %.1f%% inertia\n", cumvar_3))
cat(sprintf("✓ K-means K=3: %.1f%% between-cluster variance\n",
            100 * kmeans_result$betweenss / kmeans_result$totss))
cat(sprintf("✓ Cluster-site association: p = %.4f\n", chi_test$p.value))


# Merge with original patient features for cluster characterization
patient_with_dims <- patient_features %>%
  left_join(dim_scores %>% select(record_id, cluster, Dim1, Dim2, Dim3), 
            by = "record_id")


In [ ]:
%%R

set.seed(123) 
kmeans_result <- kmeans(dim_scores[, c("Dim1", "Dim2", "Dim3","Dim4", "Dim5")], centers = 3, nstart = 25)

dim_scores$cluster <- factor(kmeans_result$cluster)

# 3D plot for Clusters
fig_cluster <- plot_ly(dim_scores, 
  x = ~Dim1, y = ~Dim2, z = ~Dim3, 
  color = ~cluster,  
  colors = c("red", "blue", "green", "purple"), 
  type = "scatter3d", mode = "markers",
  marker = list(size = 5, opacity = 0.7),
  text = ~paste("ID:", record_id, "<br>Cluster:", cluster, "<br>Site:", site),
  hoverinfo = "text"
) %>%
  layout(
    title = "MCA Clusters in 3D",
    scene = list(
      xaxis = list(title = sprintf("Dim1 (%.1f%%)", var_explained[1, 2])),
      yaxis = list(title = sprintf("Dim2 (%.1f%%)", var_explained[2, 2])),
      zaxis = list(title = sprintf("Dim3 (%.1f%%)", var_explained[3, 2])),
      backgroundcolor = "white"  
    ),
    paper_bgcolor = "white",  
    plot_bgcolor = "lightgrey",   
    font = list(color = "black"), 
    legend = list(
      title = list(text = "Cluster"), 
      x = 0.75,
      y = 0.5,  
      xanchor = 'left', 
      yanchor = 'middle', 
      traceorder = 'normal', 
      font = list(size = 12, color = "black"), 
      bgcolor = "rgba(255, 255, 255, 0.5)",  
      bordercolor = "black", 
      borderwidth = 1 
    )
  )

fig_cluster


In [ ]:
%%R




site_colors <- c("CMH" = "#1f77b4", "NCH" = "#ff7f0e", "SK" = "#2ca02c")


dim_scores$site <- factor(dim_scores$site,
                          levels = c("SK", "CMH", "NCH"),
                          labels = c("Site 1", "Site 2", "Site 3"))


site_colors <- c("Site 1" = "#2ca02c",
                 "Site 2" = "#1f77b4",
                 "Site 3" = "#ff7f0e")

# 3D plot for Hospital Sites
fig_site <- plot_ly(dim_scores, 
  x = ~Dim1, y = ~Dim2, z = ~Dim3, 
  color = ~site, 
  colors = site_colors, 
  type = "scatter3d", mode = "markers",
  marker = list(size = 5, opacity = 0.7),
  text = ~paste("ID:", record_id, "<br>Cluster:", cluster, "<br>Site:", site),
  hoverinfo = "text"
) %>%
  layout(
    title = "MCA by Hospital Sites in 3D",
    scene = list(
      xaxis = list(title = sprintf("Dim1 (%.1f%%)", var_explained[1, 2])),
      yaxis = list(title = sprintf("Dim2 (%.1f%%)", var_explained[2, 2])),
      zaxis = list(title = sprintf("Dim3 (%.1f%%)", var_explained[3, 2])),
      backgroundcolor = "white"  
    ),
    paper_bgcolor = "white", 
    plot_bgcolor = "lightgrey",   
    font = list(color = "black"), 
    legend = list(
      title = list(text = "Hospital Site"), 
      x = 0.75, 
      y = 0.5,  
      xanchor = 'left',
      yanchor = 'middle',
      traceorder = 'normal', 
      font = list(size = 12, color = "black"), 
      bgcolor = "rgba(255, 255, 255, 0.5)", 
      bordercolor = "black", 
      borderwidth = 1 
    )
  )

fig_site


In [ ]:
%%R

# Get the variable contributions for each dimension from the MCA result
var_contrib <- mca_result$var$contrib

# Function to extract top N loadings for each dimension
get_top_loadings <- function(contrib_matrix, top_n = 10) {
  top_loadings_list <- list()
  
  # Loop through each dimension and get the top N variables
  for (dim in 1:ncol(contrib_matrix)) {
    # Get the indices of top N variables by contribution
    top_idx <- order(contrib_matrix[, dim], decreasing = TRUE)[1:top_n]
    
    # Store the variable names and their contributions
    top_loadings_list[[dim]] <- data.frame(
      Variable = rownames(contrib_matrix)[top_idx],
      Contribution = contrib_matrix[top_idx, dim]
    )
  }
  
  return(top_loadings_list)
}

# Get the top 10 loadings for each dimension
top_loadings <- get_top_loadings(var_contrib, top_n = 10)

# Print top loadings for each dimension
for (dim in 1:length(top_loadings)) {
  cat(sprintf("\n--- Top Loadings for Dimension %d ---\n", dim))
  print(top_loadings[[dim]])
}


In [ ]:
%%R

# Step 1: Extract MCA dimension scores for patients
mca_dims <- data.frame(record_id = patient_features$record_id, 
                       Dim1 = mca_result$ind$coord[, 1],
                       Dim2 = mca_result$ind$coord[, 2],
                       Dim3 = mca_result$ind$coord[, 3],
                       Dim4 = mca_result$ind$coord[, 4],
                       Dim5 = mca_result$ind$coord[, 5])

# Step 2: Add binary columns for each lesion location
significant_lesions <- c("Lumbar", "Thoracic", "Sacrum", "Iliac", "Radius", "Toe", "Finger", "Humerus",'Foot','Femur','Tibia','Fibula', 'Ankle','Knee', 'Metatarsals')

if("Lesion_location_region" %in% colnames(patient_features)) {
  for(location in significant_lesions) {
    lesion_col <- paste("has_Lesion_location_", location, sep = "")
    mca_dims[[lesion_col]] <- ifelse(patient_features$Lesion_location_region == location, 1, 0)
  }
} else {
  cat("Error: 'Lesion_location_region' column is missing from patient_features data frame.\n")
}

# Step 3: Logistic regression for each significant lesion location using MCA dimensions
cat("\n--- Logistic Regression: MCA dimensions predicting lesion locations ---\n")

for(location in significant_lesions) {
  lesion_col <- paste("has_Lesion_location_", location, sep = "")
  
  formula_str <- paste(lesion_col, "~ Dim1 + Dim2 + Dim3 + Dim4 + Dim5")
  model <- glm(as.formula(formula_str), data = mca_dims, family = binomial)
  
  cat(sprintf("\n%s: \n", location))
  summary(model) 
  
  # Compute and display Odds Ratios
  odds_ratios <- exp(coef(model))
  cat("\n--- Odds Ratios ---\n")
  print(odds_ratios) 
  
  anova_results <- anova(model, test = "Chisq")
  cat("\n--- ANOVA Results ---\n")
  print(anova_results)
}

# Step 4: Check if MCA dimensions are significantly predicting lesion locations
cat("\n--- Significant Results ---\n")
for(location in significant_lesions) {
  lesion_col <- paste("has_Lesion_location_", location, sep = "")
  model <- glm(as.formula(paste(lesion_col, "~ Dim1 + Dim2 + Dim3 + Dim4 + Dim5")), data = mca_dims, family = binomial)
  pvals <- summary(model)$coefficients[, 4]  
  cat(sprintf("\n%s:\n", location))
  print(pvals)  
}


In [ ]:
%%R
#

mca_dims <- data.frame(record_id = patient_features$record_id, 
                       Dim1 = mca_result$ind$coord[, 1],
                       Dim2 = mca_result$ind$coord[, 2],
                       Dim3 = mca_result$ind$coord[, 3],
                       Dim4 = mca_result$ind$coord[, 4],
                       Dim5 = mca_result$ind$coord[, 5])

if("Lesion_location_region" %in% colnames(patient_features)) {
  
  mca_dims$has_spinal_lesions <- ifelse(patient_features$Lesion_location_region %in% c("Thoracic", "Lumbar", "Cervical", "Sacral"), 1, 0)
  
} else {
  cat("Error: 'Lesion_location_region' column is missing from patient_features data frame.\n")
}

# Logistic regression for spinal lesions using MCA dimensions
cat("\n--- Logistic Regression: MCA dimensions predicting spinal lesions ---\n")

formula_str <- "has_spinal_lesions ~ Dim1 + Dim2 + Dim3 + Dim4 + Dim5"
model_spinal <- glm(as.formula(formula_str), data = mca_dims, family = binomial)

cat("\nSpinal Lesions:\n")
summary(model_spinal)

anova_results_spinal <- anova(model_spinal, test = "Chisq")
cat("\n--- ANOVA Results ---\n")
print(anova_results_spinal)


cat("\n--- Significant Results ---\n")
pvals_spinal <- summary(model_spinal)$coefficients[, 4] 
cat("\nSpinal Lesions:\n")
print(pvals_spinal) 


In [ ]:
%%R

mca_dims <- data.frame(record_id = patient_features$record_id, 
                       Dim1 = mca_result$ind$coord[, 1],
                       Dim2 = mca_result$ind$coord[, 2],
                       Dim3 = mca_result$ind$coord[, 3],
                       Dim4 = mca_result$ind$coord[, 4],
                       Dim5 = mca_result$ind$coord[, 5])

if("together" %in% colnames(patient_features)) {
  
  pelvic_regions <- c("Hip", "Iliac", "Sacrum", "Pubis", "Ischium", "Femur, Proximal Metaphyseal", "Femur, Proximal Epiphyseal")
  
  mca_dims$has_pelvic_lesions <- ifelse(sapply(patient_features$together, 
                                               function(x) any(sapply(pelvic_regions, grepl, x))), 1, 0)
  
} else {
  cat("Error: 'together' column is missing from patient_features data frame.\n")
}

cat("\n--- Logistic Regression: MCA dimensions predicting pelvic lesions ---\n")

formula_str <- "has_pelvic_lesions ~ Dim1 + Dim2 + Dim3 + Dim4 + Dim5"
model_pelvic <- glm(as.formula(formula_str), data = mca_dims, family = binomial)

cat("\nPelvic Lesions:\n")
summary(model_pelvic) 

cat("\n--- Odds Ratios for MCA dimensions ---\n")
odds_ratios <- exp(coef(model_pelvic)) 
print(odds_ratios)

cat("\n--- ANOVA Results ---\n")
anova_results_pelvic <- anova(model_pelvic, test = "Chisq")
print(anova_results_pelvic)

cat("\n--- P-values for each MCA dimension ---\n")
pvals_pelvic <- summary(model_pelvic)$coefficients[, 4]  
print(pvals_pelvic) 

cat("\n--- Report: MCA Dimensions and Pelvic Lesions ---\n")
for (i in 2:length(pvals_pelvic)) { 
  cat(paste0("Dimension ", i-1, ": p-value = ", round(pvals_pelvic[i], 5), 
             ", Odds Ratio = ", round(odds_ratios[i], 3), "\n"))
}
